# Homework 17.2 - Coding

This is the coding portion of the homework assignment for Section 17.2

In [12]:
import numpy as np

## Problem 17.9 

Code up a simulation of a single pull on an $n$-armed bandit system, as follows: 

Write a method called `pull` that takes the following inputs: 

* An array of $n$ probabilities $\theta_1, \ldots, \theta_n$ corresponding to the true probability of success for each arm 
* An array of payouts $(J_1, \ldots, J_n)$ for each arm 
* An action $i$ indicating that arm number $i$ should be pulled

This should return:

* The amount won 
* A pair $(\Delta a_i, \Delta b_i) \in \{(1,0),(0,1)\}$, with $(1,0)$ corresponding to a success of the $i$ th arm (the one that was pulled), and $(0,1)$ corresponding to a failure.

**HINT:** (How to draw from a bernoulli distribution) To draw from a Bernoulli distribution with probability $\theta$, you can draw $u$ from a uniform distribution on $[0,1]$ and then return success if $u \leq \theta$. Alternatively, Bernoulli is a special case of binomial, and many computational systems already have methods for drawing from a binomial distribution. 

In [13]:
def pull(
    probs: list[float], 
    payouts: list[float], 
    i:int
) -> tuple[float, tuple[int,int]]:
    """Simulate a single pull on an n-armed bandit system.
    
    Parameters:
        probs ((n,) list[float]): A list of n probabilities corresponding
            to the true probability of success for each arm 
        payouts ((n,) list[float]): A list of n payouts corresponding 
            to the payout of each arm 
        i (int): The index of the arm to pull (in {0,1,...,n-1})
    
    Returns:
        amt (float): The amount won 
        success (tuple[int,int]): (1,0) if the pull was a success, and 
            (0,1) if the pull was a failure
    """
    # n = len(probs)
    # np.random.uniform(0, 1, size=n)
    is_success = int(np.random.uniform(0, 1) <= probs[i])

    return payouts[i] * is_success, (is_success,1 - is_success)

Now, run the following code cell to visualize the results of pulling each each arm of the following multi-armed bandit:

* Arm 0: Payout: $50, Probability of Success: 50%. 
* Arm 1: Payout: $100, Probability of Success: 30%.
* Arm 2: Payout: $10, Probability of Success: 90%. 

In [14]:
### ----------------- YOU DO NOT NEED TO EDIT THIS CELL ---------------- ### 
### -------- Just run it to get results for a simple toy problem -------- ###
probs = [0.5, 0.3, 0.9]
payouts = [50., 100., 10.]
for i in range(len(probs)):
    print(f"----------------- ARM {i} -----------------")
    amt_won, success = pull(probs, payouts, i)
    print(f"Amount Won: {amt_won}")
    print(f"Success Tuple: {success}\n")

----------------- ARM 0 -----------------
Amount Won: 50.0
Success Tuple: (1, 0)

----------------- ARM 1 -----------------
Amount Won: 100.0
Success Tuple: (1, 0)

----------------- ARM 2 -----------------
Amount Won: 10.0
Success Tuple: (1, 0)



## Problem 17.10(i)

Write a method `compute_R` that accepts as input:

* $M$ (an integer)
* $r$ (a float)
* $\beta$ (a float)

and returns an $(M+1) \times (M+1)$ array `R_values` with `R_values[a,b]`$=R(a,b,r)$ for all $a+b \leq M$ (the remaining entries in the array can be set to zero)

(a) Initialize using the assumption of (17.10) for $a+b = M$

(b) Use the recursion formula (17.7) to find the other values for $1 \leq a+b < M$


In [15]:
def compute_R(
    M:int,
    r:float, 
    beta:float
) -> np.ndarray:
    """Compute the optimal reward values.

    Use (17.7) and (17.10) to compute this recursively.

    Parameters:
        M (int): The maximal amount of trials to consider
        r (float): The (undiscounted) expected value of the one-armed bandit 
        beta (float): The discount factor of the one-armed bandit 

    Returns:
        R_values ((M+1,M+1) ndarray): An array satisfying 
            R_values[a,b] = R(a,b,r)
            where R(a,b,r) is defined as in (17.7)/(17.10)
            in the textbook.
    """

    R_mat = np.zeros((M+1,M+1))

    for a in range(M, -1, -1):
        for b in range(M - a, -1, -1):
            if a + b == 0:
                continue 
            # base case
            if a + b == M:
                #  use (17.10)
                R_mat[a, b] = (1 / (1 - beta)) * max(a / (a + b), r) 
            else:
                # use (17.7)
                R_mat[a, b] = max((a * (1 + beta * R_mat[a + 1, b]) + b * beta * R_mat[a, b + 1]) / (a + b),    r / (1 - beta))

    return R_mat

**(c)** Run the following code cell to apply your code to the situation of Example 17.2.4, and compare your results to the results of that example.

In [ ]:
### ----------------- YOU DO NOT NEED TO EDIT THIS CELL ---------------- ### 
### ----- Just run it to compare results to that with the textbook ----- ###
M = 4       # Maximum number of trials 
r = 0.35    # Current expected value used in the example 
beta = 0.9  # Discount factor 

# ----- Get your code's results ----- # 
R_vals = compute_R(M, r, beta)

# ----- Compare results to that of the example ----- #
print("----------- COMPARISON TO BOOK EXAMPLES -------------")
print("Quantity\tComputed Value\t\tActual Value")
print(f"R(0,4)\t\t{R_vals[0,4]:0.2f}\t\t\t{3.5:0.2f}")
print(f"R(1,3)\t\t{R_vals[1,3]:0.2f}\t\t\t{3.5:0.2f}")
print(f"R(2,2)\t\t{R_vals[2,2]:0.2f}\t\t\t{5:0.2f}")
print(f"R(3,1)\t\t{R_vals[3,1]:0.2f}\t\t\t{7.5:0.2f}")
print(f"R(4,0)\t\t{R_vals[4,0]:0.2f}\t\t\t{10:0.2f}\n")

print(f"R(0,3)\t\t{R_vals[0,3]:0.2f}\t\t\t{3.5:0.2f}")
print(f"R(1,2)\t\t{R_vals[1,2]:0.2f}\t\t\t{3.93:0.2f}")
print(f"R(2,1)\t\t{R_vals[2,1]:0.2f}\t\t\t{6.66:0.2f}")
print(f"R(3,0)\t\t{R_vals[3,0]:0.2f}\t\t\t{10:0.2f}\n")

print(f"R(0,2)\t\t{R_vals[0,2]:0.2f}\t\t\t{3.5:0.2f}")
print(f"R(1,1)\t\t{R_vals[1,1]:0.2f}\t\t\t{5.27:0.2f}")
print(f"R(2,0)\t\t{R_vals[2,0]:0.2f}\t\t\t{10:0.2f}\n")

print(f"R(0,1)\t\t{R_vals[0,1]:0.2f}\t\t\t{3.5:0.2f}")
print(f"R(1,0)\t\t{R_vals[1,0]:0.2f}\t\t\t{10:0.2f}")

[[ 0.          3.5         3.5         3.5         3.5       ]
 [10.          5.27        3.93333333  3.5         0.        ]
 [10.          6.66666667  5.          0.          0.        ]
 [10.          7.5         0.          0.          0.        ]
 [10.          0.          0.          0.          0.        ]]
----------- COMPARISON TO BOOK EXAMPLES -------------
Quantity	Computed Value		Actual Value
R(0,4)		3.50			3.50
R(1,3)		3.50			3.50
R(2,2)		5.00			5.00
R(3,1)		7.50			7.50
R(4,0)		10.00			10.00

R(0,3)		3.50			3.50
R(1,2)		3.93			3.93
R(2,1)		6.67			6.66
R(3,0)		10.00			10.00

R(0,2)		3.50			3.50
R(1,1)		5.27			5.27
R(2,0)		10.00			10.00

R(0,1)		3.50			3.50
R(1,0)		10.00			10.00


---

IMPORTANT: Please "Restart and Run All" and ensure there are no errors. Then, submit this .ipynb file to Gradescope.